## Pipeline for 65-HD-sEMG / Hyser noise-robustness sweeps (unified)

This replaces the two separate `norm-by-channel` / `norm-by-sample` notebooks with a single
notebook parameterized by `params['norm_mode']`.

**Changes vs. the original two notebooks:**

- `norm_mode` (`'by_channel'` vs `'by_sample'`) and `dataset_name` (`'65_HDsEMG'` vs `'Hyser'`)
  are now config values (`params[...]`) instead of two separate copy-pasted notebooks. Flipping
  `norm_mode` also picks the matching model checkpoint automatically.
- **Noise is additive for every dataset.** Previously `65_HDsEMG` *replaced* a channel with pure
  noise (`channel = noise`), while only the (unused) `Hyser` branch actually added noise on top of
  the signal (`channel = channel + noise`). Both paths now go through the same `additive_noise`
  helper: a zero-mean, band-limited perturbation added to the real signal, peak-scaled to
  `max_var` times that channel's own dynamic range. "Dead electrode" is kept as a **separate**,
  explicitly non-additive fault mode (`fault_mode='dead'`), since zeroing a channel is a distinct
  failure mode from interference noise, not a variant of it.
- Fixed bugs found in the original `norm-by-sample` notebook:
  - `predict_gesture`'s signature grew a required `dataset_name` argument, but 4 of 5 sweep cells
    still called it with the old arity and would raise `TypeError`.
  - `inject_electrode_failure`'s `"Dead_electrodes"` branch referenced a loop variable `nn` that
    was never bound in that branch (no `for nn in ...`) — it only "worked" by accident because
    Jupyter's global namespace happened to still have `nn` from an unrelated earlier cell.
  - `max_var` was read from the enclosing notebook's global scope inside `inject_electrode_failure`
    instead of being passed as a parameter.
  - The hand-built `index1`/`index2` channel-index arrays were only 64 elements long. The
    "sweep all channels" cell in the original `norm-by-channel` notebook crashes with
    `IndexError: index 64 is out of bounds for axis 0 with size 64` once the sweep asks for more
    than 64 noisy channels (visible in that notebook's own saved output). Replaced with
    `build_channel_order`, sized to the actual channel count of the loaded data.
- The model is now loaded **once** and reused across the whole sweep, instead of calling
  `tf.keras.models.load_model` inside `predict_gesture` on every single `(max_var, n_channels)`
  combination (this was the dominant cost of each ~130s sweep point in the original logs).
- `model.predict(x)` (single-sample) replaced with a direct `model(x, training=False)` call to
  avoid `Model.predict`'s per-call dataset-building overhead in the online/streaming loop.
- Noise generation is now driven by a seeded `numpy.random.Generator` and each sweep point is run
  `params['n_trials']` times and averaged (mean/std reported). The original single-draw-per-point
  sweeps produced visibly non-monotonic accuracy-vs-#channels curves (e.g. accuracy going *up* as
  more channels were corrupted) that were most likely just noise-draw variance, not a real trend.
- The ~40-line data-loading/gesture-selection block that was duplicated 5-6 times per notebook is
  now `load_subject_gesture_data(...)`, and the 5 duplicated sweep blocks are now one
  `run_experiment(...)` — this is exactly the kind of drift that caused the `dataset_name` bug
  above, since one call site can silently fall behind when there's no shared implementation.

**Not run here.** `ProcessingPipeline`, `EMGGestureDataGen`, `KerasDataPipeline`,
`EMGConv2DResNetModel`, `utils`, `available_metrics`, the `trainings/` checkpoints, and the
`/root/data/data/65_HDsEMG/` dataset are not present in this environment, so this notebook has
**not** been executed end-to-end. Please run it (ideally starting with a tiny sweep, e.g. 1-2
`(max_var, n_channels)` points) in your normal environment before trusting the full sweep or the
CSV outputs.


In [ ]:
import os, sys

processing_source_path = os.path.abspath('./../../../Processing/')
if(processing_source_path not in sys.path):
    sys.path.append(processing_source_path)
import ProcessingPipeline as pp

dataloader_source_path = os.path.abspath('./../../../Dataloader/')
if(dataloader_source_path not in sys.path):
    sys.path.append(dataloader_source_path)
from EMGGestureDataGen import EMGGestureDataGen
from KerasDataPipeline import KerasDataPipeline

model_source_path = os.path.abspath('./../../../Keras_deep_models/')
if(model_source_path not in sys.path):
    sys.path.append(model_source_path)
from EMGConv2DResNetModel import EMGConv2DResNetModel

utils_source_path = os.path.abspath('./../../../Utils/')
if(utils_source_path not in sys.path):
    sys.path.append(utils_source_path)
import utils as utils
import available_metrics as metrics

In [ ]:
import tensorflow as tf
import sklearn.metrics as skm
import numpy as np
import glob
import time
import gc
import h5py
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)

## Configuration

Flip `norm_mode` / `dataset_name` here to switch pipeline variant instead of maintaining separate notebooks.

In [ ]:
# Maps the human-readable norm_mode to the axis `norm()` should reduce over, and to the
# filename suffix used by the two matching model checkpoints.
NORM_AXIS_BY_MODE = {'by_channel': 1, 'by_sample': 0}
NORM_SUFFIX_BY_MODE = {'by_channel': 'bychannel', 'by_sample': 'bysample'}
HD65_EMG_GESTURE_LIST = [0, 13, 14, 17, 20, 36, 58, 59, 63, 64]
HYSER_GESTURE_LIST = [0, 2, 20, 21, 26, 27, 30 ,31, 33, 34]


params = {
    'dataset_path': '/root/data/data/65_HDsEMG/',
    'dataset_name': '65_HDsEMG',   # '65_HDsEMG' (bipolar/differential channels) or 'Hyser' (monopolar)
    'norm_mode': 'by_channel',     # 'by_channel' or 'by_sample' -- selects norm() axis + model checkpoint
    'window': 128,
    'step': 32,
    'num_channels': 64,
    'num_classes': 9,
    'num_samples': 256,
    'num_windows': 10,
    'img_height': 8,
    'img_width': 16,
    'print_model': True,
    'num_epochs': 60,
    'gesture_list': HD65_EMG_GESTURE_LIST,  # 65-HD-sEMG list
    'n_voting': 3,
    'armband': False,
    'MAFilter': 32,
    'plot_data': False,
    'noise_seed_base': 0,          # trial `t` uses seed noise_seed_base + t, for reproducibility
    'n_trials': 3,                 # repeats per sweep point; accuracy is reported as mean +/- std
}

model_name_template = "trainings/EMG_65HDsEMG_2D_CNN_128f_128ch_SCE_normminmax_{suffix}_adamW"
model_name = model_name_template.format(suffix=NORM_SUFFIX_BY_MODE[params['norm_mode']])
model_name

## Buffering data

In [ ]:
def buffer_data(data, new_batch):
    data = np.roll(data, -new_batch.shape[1], axis=1)
    data[:, -new_batch.shape[1]:] = new_batch
    return data

## Preprocessing

In [ ]:
def EMGTKE(data):
    return data[:, 1:-1] * data[:, 1:-1] - data[:, :-2] * data[:, 2:]

In [ ]:
def retification(data):
    return np.abs(data)

In [ ]:
def MAFilter(data, L):
    cs = np.cumsum(data, axis=1, dtype=float)
    cs[:, L:] = cs[:, L:] - cs[:, :-L]
    return cs[:, L - 1:] / L

In [ ]:
def norm(data, ax):
    samples_min = data.min(axis=ax, keepdims=True)
    data = (data - samples_min)
    samples_max = data.max(axis=ax, keepdims=True) + sys.float_info.epsilon
    data = data / samples_max
    return data

## Noise / fault generator

`additive_noise` is the single noise model used for **every** dataset (`apply_electrode_noise`
below picks the right domain to add it in, but the noise itself and its amplitude scaling are
identical). `apply_dead_electrodes` is kept separate on purpose: it is a distinct fault type
(permanent channel loss), not a noise variant.

In [ ]:
def fftnoise(f, rng):
    f = np.array(f, dtype='complex')
    Np = (len(f) - 1) // 2
    phases = rng.uniform(0, 2 * np.pi, Np)
    phases = np.cos(phases) + 1j * np.sin(phases)
    f[1:Np + 1] *= phases
    f[-1:-1 - Np:-1] = np.conj(f[1:Np + 1])
    return np.fft.ifft(f).real

In [ ]:
def band_limited_noise(min_freq, max_freq, samples, samplerate, rng):
    freqs = np.abs(np.fft.fftfreq(samples, 1 / samplerate))
    f = np.zeros(samples)
    idx = np.where(np.logical_and(freqs >= min_freq, freqs <= max_freq))[0]
    f[idx] = 1
    return fftnoise(f, rng)

In [ ]:
def additive_noise(channel_data, max_var, rng, min_freq=10, max_freq=900, samplerate=2048):
    """
    Zero-mean, band-limited perturbation meant to be ADDED to a channel, not to replace it.
    Peak-scaled to `max_var` times that channel's own (min, max) dynamic range, so amplitude is
    comparable across channels/datasets regardless of their absolute signal scale.
    """
    data_len = channel_data.shape[-1]
    span = np.max(channel_data) - np.min(channel_data)
    x = band_limited_noise(min_freq, max_freq, data_len, samplerate, rng)
    x = x - x.mean()
    peak = np.max(np.abs(x)) + sys.float_info.epsilon
    return x / peak * (span * max_var / 2)

In [ ]:
def build_channel_order(num_channels, img_height, img_width, shuffle_seed=None):
    """
    Deterministic list of `num_channels` channel indices, used to decide which channels get
    corrupted first as a sweep's `n_channels` grows. Replaces the old hand-built `index1`/`index2`
    arrays, which were only long enough for 64 channels and raised IndexError once a sweep asked
    for more noisy channels than that.

    Order follows a column-major traversal of the electrode grid, so consecutive channels come
    from spatially spread-out electrodes rather than one contiguous block.
    """
    grid_size = img_height * img_width
    if grid_size < num_channels:
        raise ValueError(f"grid {img_height}x{img_width}={grid_size} smaller than num_channels={num_channels}")
    order = np.arange(grid_size).reshape(img_height, img_width).flatten(order='F')
    order = order[order < num_channels]
    if shuffle_seed is not None:
        np.random.default_rng(shuffle_seed).shuffle(order)
    return order

In [ ]:
def apply_electrode_noise(data, channel_order, n_channels, max_var, rng, dataset_name):
    """
    Additive noise fault, injected the same way for every dataset.

    For '65_HDsEMG' the raw channels are bipolar/differential; noise is added in the reconstructed
    monopolar domain (cumulative sum) and the signal is differentiated back, so one noisy electrode
    still shows up correctly on both of its neighbouring differential channels, same as it would on
    real hardware. For every other dataset (e.g. 'Hyser') channels are already monopolar and the
    noise is added directly.
    """
    data = data.copy()
    if dataset_name == '65_HDsEMG':
        for nn in range(n_channels):
            ch = channel_order[nn]
            noise = additive_noise(data[ch, :], max_var, rng)
            data[ch, :] = data[ch, :] + noise
            data[ch+1, :] = data[ch+1, :] - noise
    else:
        for nn in range(n_channels):
            ch = channel_order[nn]
            data[ch, :] = data[ch, :] + additive_noise(data[ch, :], max_var, rng)
    return data

In [ ]:
def apply_dead_electrodes(data, channel_order, n_channels, dataset_name):
    """Separate fault mode (not noise): permanently zero out `n_channels` electrodes."""
    data = data.copy()
    if dataset_name == '65_HDsEMG':
        mono = np.cumsum(data, axis=0)
        for nn in range(n_channels):
            mono[channel_order[nn], :] = 0
        data = np.diff(np.vstack([np.zeros((1, mono.shape[1])), mono]), axis=0)
    else:
        for nn in range(n_channels):
            data[channel_order[nn], :] = 0
    return data

## Data loading

Single helper replacing the ~40-line block that was duplicated 5-6 times per original notebook.

In [ ]:
def load_subject_gesture_data(file_path, gest, params):
    with h5py.File(file_path, 'r') as hf:
        extensor = hf['emg_extensors'][...].reshape(64, -1).astype(np.float32)
        flexor = hf['emg_flexors'][...].reshape(64, -1).astype(np.float32)
        label = hf['adjusted_class'][...]

    emg_data = np.concatenate((extensor, flexor), axis=0)
    if params['armband']:
        idx = [np.arange(0, 121, 8), np.arange(1, 58, 8), np.arange(2, 59, 8),
               np.arange(5, 62, 8), np.arange(6, 63, 8), np.arange(7, 129, 8)]
        idx = np.hstack(idx)
        emg_data = np.delete(emg_data, idx, axis=0)

    gesture_idx = np.argwhere(label[0, :] == gest)[:, 0]
    emg_data = emg_data[:, gesture_idx]
    gesture_labels = label[:, gesture_idx][0]

    all_gestures = np.sort(params['gesture_list'])
    gesture_remap = {g: new_g for new_g, g in enumerate(all_gestures)}
    yData = np.array([gesture_remap[g] for g in gesture_labels])

    return emg_data, yData

## Predict patterns

In [ ]:
def predict_gesture(model, temp_data, y, params, channel_order, n_channels, max_var, fault_mode, rng):
    """
    fault_mode: 'none' | 'noise' | 'dead'.
    `model` must already be loaded (see the experiment cells below) -- reloading it on every call,
    as the original notebooks did, was the dominant cost of each sweep point.
    """
    if fault_mode not in ('none', 'noise', 'dead'):
        raise ValueError(f"unknown fault_mode: {fault_mode}")

    y_true, y_pred = [], []
    t_init = time.time()
    all_latent_times = []
    t_lat_mean = t_init
    latent_times = {'first_bacth': 0, 'second_bacth': 0, 'mean': 0}

    temp_data = temp_data.copy()
    if fault_mode == 'noise' and n_channels > 0:
        temp_data = apply_electrode_noise(temp_data, channel_order, n_channels, max_var, rng, params['dataset_name'])
    elif fault_mode == 'dead' and n_channels > 0:
        temp_data = apply_dead_electrodes(temp_data, channel_order, n_channels, params['dataset_name'])

    data = np.zeros((params['img_height'] * params['img_width'], params['num_samples']))
    data_prev = data[:, -params['MAFilter']:].copy()
    y_cnn = np.zeros((params['n_voting']))
    norm_axis = NORM_AXIS_BY_MODE[params['norm_mode']]

    for i in range(0, params['num_windows'] * params['step'], params['step']):
        new_data = retification(temp_data[:, i:i + params['step']])
        data_MA = np.concatenate((data_prev, new_data), axis=1)
        data_prev = data_MA[:, -params['MAFilter']:].copy()

        data_MA = EMGTKE(data_MA)
        data_MA = MAFilter(data_MA, params['MAFilter'])
        data = buffer_data(data, data_MA)

        data_prep = norm(data, norm_axis)
        data_prep = data_prep.reshape(1, params['img_width'], params['img_height'], params['num_samples'])

        if params['plot_data']:
            plt.plot(data_prep[0, 0, 0, :])
            plt.title('after norm')
            plt.show()

        y_cnn = np.roll(y_cnn, -1)
        y_cnn[-1] = np.argmax(model(data_prep, training=False).numpy())

        if len(y_pred) < params['n_voting']:
            y_pred += [y_cnn[-1]]
        else:
            values, count = np.unique(y_cnn, return_counts=True)
            if len(count) == params['n_voting']:
                y_pred += [y_pred[-1]]
            else:
                y_pred += [values[np.argmax(count)]]

        values, count = np.unique(y[i:i + params['window']], return_counts=True)
        y_true += [values[np.argmax(count)]]

        if i == 0:
            latent_times['first_bacth'] = time.time() - t_init
            t_lat = time.time()
        elif i == params['step']:
            latent_times['second_bacth'] = time.time() - t_lat

        all_latent_times += [time.time() - t_lat_mean]
        t_lat_mean = time.time()

    latent_times['mean'] = np.mean(all_latent_times)
    return y_pred, y_true, latent_times

## Experiment runner

Single reusable sweep loop replacing the 5 duplicated ~60-line blocks from the original notebooks.

In [ ]:
def run_experiment(model, emg_data, yData, params, gest, channel_order, sweep, fault_mode, label):
    """
    sweep: iterable of (max_var, n_channels) tuples.
    Runs params['n_trials'] repeats per point with independent RNG seeds and reports mean/std
    accuracy -- averaging removes the single-random-draw variance that made the original
    accuracy-vs-#channels curves non-monotonic.
    """
    results = []
    t_init = time.time()
    for max_var, nch in sweep:
        accs = []
        latent_times = None
        for trial in range(params['n_trials']):
            rng = np.random.default_rng(params['noise_seed_base'] + trial)
            y_pred, y_true, latent_times = predict_gesture(
                model, emg_data, yData, params, channel_order, nch, max_var, fault_mode, rng)
            accs.append(skm.accuracy_score(y_true, y_pred))

        print(f"fault={fault_mode} max_var={max_var} n_channels={nch} "
              f"acc={np.mean(accs):.4f}+-{np.std(accs):.4f}")

        results.append({
            "gesture": gest,
            "fault_mode": fault_mode,
            "max_noise_amplitude": max_var,
            "num_channels": nch,
            "accuracy_mean": np.mean(accs),
            "accuracy_std": np.std(accs),
            "n_trials": params['n_trials'],
            "latency_first": latent_times['first_bacth'],
            "latency_second": latent_times['second_bacth'],
            "latency_avg": latent_times['mean'],
            "total_time": time.time() - t_init,
        })
        gc.collect()

    df = pd.DataFrame(results)
    df.to_csv(f"{label}.csv", index=False)
    return df

## Read all files

In [ ]:
files_list = glob.glob(params['dataset_path'] + '/2*')
files_list.sort()
files_list

## Model EMG

### Running on all data

### Not all of them were used on training

Subjects used on training:
+ 25295225 (2),
+ 25295423 (3),
+ 25296644 (4),
+ 25313078 (9),
+ 25313597 (11),
+ 2532065 (13),
+ 25323134 (15),
+ 25323215 (16)

### Load the model once, and load one subject/gesture

Both are reused across every sweep below instead of being reloaded per sweep point.

In [ ]:
model = tf.keras.models.load_model(model_name, compile=False)

file = files_list[1]
gest = params['gesture_list'][4]

emg_data, yData = load_subject_gesture_data(file, gest, params)
params['num_windows'] = (emg_data.shape[1] - params['step']) // params['step']
channel_order = build_channel_order(emg_data.shape[0], params['img_height'], params['img_width'])

print('num windows', params['num_windows'], 'num channels', emg_data.shape[0])
print(np.min(emg_data), np.max(emg_data))

### Experiment 1 -- additive noise, small channel counts (0-7)

Directly comparable to the original notebooks' first sweep.

In [ ]:
sweep_1 = [(round(max_var / 10, 1), nch) for max_var in range(8, 13) for nch in range(0, 8)]

df_exp1 = run_experiment(model, emg_data, yData, params, gest, channel_order,
                          sweep=sweep_1, fault_mode='noise',
                          label=f"{model_name}_experiment_1_noise")
df_exp1

### Experiment 2 -- additive noise, full channel-count sweep

`channel_order` is sized to `emg_data.shape[0]`, so this no longer raises `IndexError` past 64 channels the way the original notebook did.

In [ ]:
sweep_2 = [(round(max_var / 10, 1), nch) for max_var in range(8, 13)
           for nch in range(8, emg_data.shape[0], 8)]

df_exp2 = run_experiment(model, emg_data, yData, params, gest, channel_order,
                          sweep=sweep_2, fault_mode='noise',
                          label=f"{model_name}_experiment_2_noise_wide")
df_exp2

### Experiment 3 -- dead-electrode fault (not noise), small channel counts

Kept as an explicitly separate, non-additive fault mode.

In [ ]:
sweep_3 = [(0.0, nch) for nch in range(0, 8)]

df_exp3 = run_experiment(model, emg_data, yData, params, gest, channel_order,
                          sweep=sweep_3, fault_mode='dead',
                          label=f"{model_name}_experiment_3_dead")
df_exp3

### Experiment 4 -- experiment 1 sweep with `n_voting=5`

In [ ]:
params['n_voting'] = 5

df_exp4 = run_experiment(model, emg_data, yData, params, gest, channel_order,
                          sweep=sweep_1, fault_mode='noise',
                          label=f"{model_name}_experiment_4_noise_voting5")

params['n_voting'] = 3  # restore default
df_exp4

### Experiment 5 -- clean-signal baseline across the full gesture list, armband layout

In [ ]:
params_arm = dict(params, armband=True, n_voting=5)

results_list = []
for g in params['gesture_list']:
    emg_data_g, yData_g = load_subject_gesture_data(file, g, params_arm)
    params_arm['num_windows'] = (emg_data_g.shape[1] - params_arm['step']) // params_arm['step']
    channel_order_g = build_channel_order(emg_data_g.shape[0], params_arm['img_height'], params_arm['img_width'])

    df_g = run_experiment(model, emg_data_g, yData_g, params_arm, g, channel_order_g,
                           sweep=[(0.0, 0)], fault_mode='none',
                           label=f"{model_name}_experiment_5_gesture_{g}")
    results_list.append(df_g)

df_exp5 = pd.concat(results_list, ignore_index=True)
df_exp5.to_csv(f"{model_name}_experiment_5.csv", index=False)
print('mean accuracy across gestures:', df_exp5['accuracy_mean'].mean())
df_exp5